In [ ]:

# Install required packages (Run if necessary)
!pip install shap grad-cam tf-explain


In [ ]:
!pip install seaborn

In [ ]:
!pip install tensorflow

In [ ]:

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import shap

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tf_explain.core.grad_cam import GradCAM

print("TensorFlow Version:", tf.__version__)


In [ ]:
!pip install tensorflow

In [ ]:

# Create output folders

OUTPUT_DIR = "outputs"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Folders created successfully.")


In [ ]:

# Dataset path

DATASET_PATH = r"C:\Users\john5\OneDrive\Desktop\2026\UE SEMESTER 1\Pattern Recognition\FinalProjectGIT\Livestock_disease_image_classifier_CNN\dts\Cows datasets"


IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

CLASS_NAMES = list(train_generator.class_indices.keys())

print(CLASS_NAMES)


In [ ]:

# Dataset Description

dataset_description = pd.DataFrame({
    "Feature": ["Dataset Name", "Number of Classes", "Image Size", "Classes"],
    "Value": [
        "Cattle Disease Dataset",
        len(CLASS_NAMES),
        str(IMG_SIZE),
        ", ".join(CLASS_NAMES)
    ]
})

dataset_description.to_csv(os.path.join(TABLE_DIR, "dataset_description.csv"), index=False)

dataset_description


In [ ]:

# Display sample images

fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(15,5))

for i, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATASET_PATH, class_name)
    img_name = os.listdir(class_dir)[0]
    img_path = os.path.join(class_dir, img_name)

    img = plt.imread(img_path)

    axes[i].imshow(img)
    axes[i].set_title(class_name)
    axes[i].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "dataset_samples.pdf"))
plt.show()


In [ ]:

# Custom CNN

custom_cnn = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(len(CLASS_NAMES), activation='softmax')
])

custom_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

custom_cnn.summary()


In [ ]:

base_mobilenet = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_mobilenet.trainable = False

x = base_mobilenet.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(len(CLASS_NAMES), activation='softmax')(x)

mobilenet_model = Model(inputs=base_mobilenet.input, outputs=output)

mobilenet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mobilenet_model.summary()


In [ ]:

base_resnet = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_resnet.trainable = False

x = base_resnet.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(len(CLASS_NAMES), activation='softmax')(x)

resnet_model = Model(inputs=base_resnet.input, outputs=output)

resnet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

resnet_model.summary()


In [ ]:

# Training function

EPOCHS = 10

def train_model(model, model_name):

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS
    )

    model.save(os.path.join(MODEL_DIR, f"{model_name}.h5"))

    # Accuracy plot
    plt.figure(figsize=(8,5))
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.legend()
    plt.title(f"{model_name} Accuracy")
    plt.savefig(os.path.join(FIG_DIR, f"{model_name}_accuracy.pdf"))
    plt.close()

    # Loss plot
    plt.figure(figsize=(8,5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.legend()
    plt.title(f"{model_name} Loss")
    plt.savefig(os.path.join(FIG_DIR, f"{model_name}_loss.pdf"))
    plt.close()

    return history


In [ ]:

# Train all models

history_custom = train_model(custom_cnn, "Custom_CNN")
history_mobile = train_model(mobilenet_model, "MobileNetV2")
history_resnet = train_model(resnet_model, "ResNet50")


In [ ]:

# Evaluation Function

def evaluate_model(model, model_name):

    predictions = model.predict(val_generator)
    y_pred = np.argmax(predictions, axis=1)
    y_true = val_generator.classes

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    results = pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
        "Value": [acc, precision, recall, f1]
    })

    results.to_csv(os.path.join(TABLE_DIR, f"{model_name}_metrics.csv"), index=False)

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        output_dict=True
    )

    pd.DataFrame(report).transpose().to_csv(
        os.path.join(TABLE_DIR, f"{model_name}_classification_report.csv")
    )

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{model_name} Confusion Matrix")

    plt.savefig(os.path.join(FIG_DIR, f"{model_name}_confusion_matrix.pdf"))
    plt.close()

    return results


In [ ]:

# Evaluate models

custom_results = evaluate_model(custom_cnn, "Custom_CNN")
mobile_results = evaluate_model(mobilenet_model, "MobileNetV2")
resnet_results = evaluate_model(resnet_model, "ResNet50")


In [ ]:

# Comparative Results Table

comparison_df = pd.DataFrame({
    "Model": ["Custom CNN", "MobileNetV2", "ResNet50"],
    "Accuracy": [
        custom_results.iloc[0,1],
        mobile_results.iloc[0,1],
        resnet_results.iloc[0,1]
    ],
    "Precision": [
        custom_results.iloc[1,1],
        mobile_results.iloc[1,1],
        resnet_results.iloc[1,1]
    ],
    "Recall": [
        custom_results.iloc[2,1],
        mobile_results.iloc[2,1],
        resnet_results.iloc[2,1]
    ],
    "F1-Score": [
        custom_results.iloc[3,1],
        mobile_results.iloc[3,1],
        resnet_results.iloc[3,1]
    ]
})

comparison_df.to_csv(os.path.join(TABLE_DIR, "comparison_results.csv"), index=False)

comparison_df


In [ ]:

# Grad-CAM Visualization

def generate_gradcam(model, image, class_index, layer_name, save_name):

    data = ([image], None)

    explainer = GradCAM()

    grid = explainer.explain(
        data,
        model,
        class_index=class_index,
        layer_name=layer_name
    )

    plt.figure(figsize=(6,6))
    plt.imshow(grid)
    plt.axis('off')

    plt.savefig(os.path.join(FIG_DIR, save_name))
    plt.close()


In [ ]:

# Example Grad-CAM generation

sample_batch = next(val_generator)
sample_image = sample_batch[0][0]
sample_label = np.argmax(sample_batch[1][0])

generate_gradcam(
    mobilenet_model,
    sample_image,
    sample_label,
    "Conv_1",
    "MobileNetV2_GradCAM.pdf"
)


In [ ]:

# SHAP Explainability

background = next(train_generator)[0][:50]

explainer = shap.DeepExplainer(mobilenet_model, background)

sample_images = next(val_generator)[0][:5]

shap_values = explainer.shap_values(sample_images)

shap.image_plot(shap_values, sample_images, show=False)

plt.savefig(os.path.join(FIG_DIR, "SHAP_Explanations.pdf"))
plt.close()


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # 1. Create a model that maps the input image to the activations of the last conv layer
    # as well as the output predictions
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs],
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    # 2. Compute the gradient of the top predicted class for our input image
    # with respect to the activations of the last conv layer
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # 3. This is the gradient of the output neuron with regard to the output feature map of the last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # 4. This is a vector where each entry is the mean intensity of the gradient over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 5. We multiply each channel in the feature map array by "how important this channel is"
    # with regard to the top predicted class, then sum all the channels to obtain the heatmap class activation
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # 6. For visualization purpose, we will also normalize the heatmap between 0 & 1
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

# --- HOW TO USE IT ---
# 1. Find your last convolutional layer name using: model.summary()
# 2. Generate the heatmap:
# heatmap = make_gradcam_heatmap(your_preprocessed_image, model, "name_of_last_conv_layer")

# 3. Plot it:
# plt.matshow(heatmap)
# plt.show()

In [ ]:

# Model Architecture Summary

architecture_df = pd.DataFrame({
    "Model": ["Custom CNN", "MobileNetV2", "ResNet50"],
    "Type": [
        "CNN from scratch",
        "Transfer Learning",
        "Transfer Learning"
    ],
    "Input Size": [
        "224x224x3",
        "224x224x3",
        "224x224x3"
    ]
})

architecture_df.to_csv(
    os.path.join(TABLE_DIR, "model_architecture.csv"),
    index=False
)

architecture_df


In [ ]:

# Zip all outputs

ZIP_NAME = "Cattle_Disease_Project_Output.zip"

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zipf:

    # Add notebook
    zipf.write("Cattle_Disease_Classification.ipynb")

    # Add outputs
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            filepath = os.path.join(root, file)
            zipf.write(filepath)

print(f"{ZIP_NAME} created successfully.")
